In [ ]:
import pandas as pd
import faiss
from models.embedder import embed_query
from src.utils import load_image
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

In [ ]:
df = pd.read_csv("data/metadata.csv")
index=faiss.read_index("data/index.faiss")
print(df.head(10))

In [ ]:
def get_query_vector(query):
    vec=embed_query(query)
    return np.array([vec]).astype("float32")


In [ ]:
def search(query, k=5):
    q = get_query_vector(query)
    distances, indices = index.search(q, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        row = df.iloc[idx]
        results.append({
            "image_path": row["image_path"],
            "class_name": row["class_name"],
            "distance": float(dist)
        })

    return results

In [ ]:
def show_results(query, results):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, len(results)+1, 1)
    if isinstance(query, str) and os.path.exists(query):
        plt.imshow(Image.open(query))
        plt.title("Query Image")
    else:
        plt.text(0.5, 0.5, str(query), ha='center', va='center')
        plt.title("Query Text")
    plt.axis('off')

    for i, r in enumerate(results):
        print(type(r))  # DEBUG LINE
        plt.subplot(1, len(results)+1, i+2)
        plt.imshow(Image.open(r["image_path"]))
        plt.title(f"{r['class_name']}\n{r['distance']:.2f}")
        plt.axis('off')

    plt.show()

In [ ]:
results = search("data/intel/seg_test/buildings/20057.jpg")
show_results("data/intel/seg_test/buildings/20057.jpg", results)

In [ ]:
results = search("a city with tall buildings")
show_results("a city with tall buildings", results)

In [ ]:
results = search("a hiker on green hills")
show_results("a hiker on green hills", results)

In [ ]:
results = search("rural")
show_results("rural", results)

In [ ]:
results = search("unusual")
show_results("unusual", results)

In [ ]:
results = search("an alien")
show_results("an alien", results)

In [ ]:
print(type(index))